# Mesmer + Segmentation Pipeline (Model Side)


In [ ]:
# ============================================================
# CELL 1 — Imports + Paths
# ============================================================
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from deepcell.applications import Mesmer
import os

DATA_DIR = "/Users/mehdi/Desktop/Computational Genomics Project/data"   # <-- same folder as Part 1
OUT_DIR  = DATA_DIR

app = Mesmer()
print("Mesmer loaded. ✓")

In [ ]:
# ============================================================
# CELL 2 — Load Your Tissue Image
# ============================================================
# Shape must be (1, H, W, 2)
# Channel 0 = nuclear marker (DAPI)
# Channel 1 = membrane/cytoplasm marker

# --- Option A: .npy ---
image = np.load(f"{DATA_DIR}/your_image.npy")

# --- Option B: two tiff channels ---
# from skimage.io import imread
# nuclear   = imread(f"{DATA_DIR}/dapi.tif")
# membrane  = imread(f"{DATA_DIR}/membrane.tif")
# image = np.stack([nuclear, membrane], axis=-1)[np.newaxis]  # (1, H, W, 2)

print("Image shape:", image.shape)   # confirm (1, H, W, 2)
print("dtype:", image.dtype)

In [ ]:
# ============================================================
# CELL 3 — Run Mesmer Segmentation
# ============================================================
segmentation = app.predict(
    image,
    image_mpp=0.5,            # <-- adjust to your dataset's microns per pixel
    compartment="whole-cell"  # or "nuclear"
)

seg_mask = segmentation[0, :, :, 0]   # (H, W), each pixel = cell ID (0 = background)
print("Cells detected:", len(np.unique(seg_mask)) - 1)

np.save(f"{OUT_DIR}/segmentation_mask.npy", seg_mask)
print("Saved segmentation_mask.npy ✓")

In [ ]:
# ============================================================
# CELL 4 — Visualize Segmentation
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(image[0, :, :, 0], cmap="gray")
axes[0].set_title("Nuclear Channel")
axes[0].axis("off")

axes[1].imshow(image[0, :, :, 1], cmap="gray")
axes[1].set_title("Membrane Channel")
axes[1].axis("off")

axes[2].imshow(seg_mask, cmap="nipy_spectral")
axes[2].set_title(f"Segmentation ({len(np.unique(seg_mask))-1} cells)")
axes[2].axis("off")

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/segmentation_overview.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved segmentation_overview.png ✓")

In [ ]:
# ============================================================
# CELL 5 — Load Spatial Data + Gene List from Person 1
# ============================================================
adata_sp = sc.read_h5ad(f"{OUT_DIR}/adata_sp_processed.h5ad")

coords = adata_sp.obsm["spatial"]   # (n_spots, 2) pixel coords
X_sp   = adata_sp.X.toarray() if hasattr(adata_sp.X, "toarray") else np.array(adata_sp.X)

with open(f"{OUT_DIR}/shared_genes.txt", "r") as f:
    shared_genes = [line.strip() for line in f.readlines()]

assert list(adata_sp.var_names) == shared_genes, "Gene order mismatch!"
print(f"Loaded {len(shared_genes)} genes, {coords.shape[0]} spots. ✓")

In [ ]:
# ============================================================
# CELL 6 — Assign Spots to Segmented Cells
# ============================================================
cell_ids = []
for (x, y) in coords:
    xi, yi = int(round(x)), int(round(y))
    if 0 <= yi < seg_mask.shape[0] and 0 <= xi < seg_mask.shape[1]:
        cell_ids.append(seg_mask[yi, xi])
    else:
        cell_ids.append(0)

cell_ids = np.array(cell_ids)
print(f"Assigned: {np.sum(cell_ids > 0)} spots | Background: {np.sum(cell_ids == 0)} spots")

In [ ]:
# ============================================================
# CELL 7 — Build Cell × Gene Expression Matrix
# ============================================================
unique_cells = np.unique(cell_ids)
unique_cells = unique_cells[unique_cells > 0]

n_cells = len(unique_cells)
n_genes = len(shared_genes)
print(f"Building matrix: {n_cells} cells × {n_genes} genes ...")

cell_expression = np.zeros((n_cells, n_genes), dtype=np.float32)
for i, cid in enumerate(unique_cells):
    cell_expression[i] = X_sp[cell_ids == cid].sum(axis=0)

print("Done. Shape:", cell_expression.shape)

In [ ]:
# ============================================================
# CELL 8 — Visualize Assignment + Save
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(seg_mask, cmap="nipy_spectral", alpha=0.7)
axes[0].scatter(coords[cell_ids > 0, 0], coords[cell_ids > 0, 1],
                c="lime", s=2, label="Assigned", alpha=0.5)
axes[0].scatter(coords[cell_ids == 0, 0], coords[cell_ids == 0, 1],
                c="red",  s=2, label="Background", alpha=0.5)
axes[0].set_title("Spot Assignment")
axes[0].legend(markerscale=5)
axes[0].axis("off")

axes[1].hist(cell_expression.sum(axis=1), bins=50, color="steelblue", edgecolor="white")
axes[1].set_xlabel("Total transcripts per cell")
axes[1].set_ylabel("Cell count")
axes[1].set_title("Transcript Distribution")

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/spot_assignment.png", dpi=150, bbox_inches="tight")
plt.show()

np.save(f"{OUT_DIR}/cell_expression.npy", cell_expression)
np.save(f"{OUT_DIR}/cell_ids_order.npy", unique_cells)

print("Saved: cell_expression.npy, cell_ids_order.npy, spot_assignment.png")
print("\nPerson 2 complete! ✓")